# Scratch Work for Flow Matching model 

In [2]:
from ase.io import read
from ase import Atoms
from ase.visualize import view

import torch 
from torch import optim
from torch import nn
from torch_geometric.data import Data
from torch import Tensor

from models.gemnet.gemnet_flow import GemNetFlow
from models.gemnet.gemnet_base import GemNetOC
from ase.io import read


ModuleNotFoundError: No module named 'models'

In [2]:
contcar = read("POSCAR.vasp")
poscar = read("POSCAR0.vasp")
# Concvert atomic positions to data object
positions = torch.tensor(contcar.get_positions(), dtype=torch.float32)
atomic_numbers = torch.tensor(contcar.get_atomic_numbers(), dtype=torch.long)

## Compile GemNet for flow matching

In [3]:
# GemNet-OC params
model_cfg = {
    "num_targets": 1,
    "num_spherical": 7,
    "num_radial": 128,
    "num_blocks": 4,
    "emb_size_atom": 256,
    "emb_size_edge": 512,
    "emb_size_trip_in": 64,
    "emb_size_trip_out": 64,
    "emb_size_quad_in": 32,
    "emb_size_quad_out": 32,
    "emb_size_aint_in": 64,
    "emb_size_aint_out": 64,
    "emb_size_rbf": 16,
    "emb_size_cbf": 16,
    "emb_size_sbf": 32,
    "num_before_skip": 2,
    "num_after_skip": 2,
    "num_concat": 1,
    "num_atom": 3,
    "num_output_afteratom": 3,
    "num_atom_emb_layers": 0,
    "num_global_out_layers": 2,
    "flow_matching": True,
    "dummy": True,
    "use_pbc": True,
    "scale_backprop_forces": False,
    "cutoff": 12.0,
    "cutoff_qint": 12.0,
    "cutoff_aeaint": 12.0,
    "cutoff_aint": 12.0,
    "max_neighbors": 30,
    "max_neighbors_qint": 8,
    "max_neighbors_aeaint": 20,
    "max_neighbors_aint": 1000,
    "enforce_max_neighbors_strictly": True,
    "rbf": {"name": "gaussian"},
    "rbf_spherical": None,
    "envelope": {"name": "polynomial", "exponent": 5},
    "cbf": {"name": "spherical_harmonics"},
    "sbf": {"name": "legendre_outer"},
    "extensive": True,
    "forces_coupled": True,
    "output_init": "HeOrthogonal",
    "activation": "silu",
    "scale_file": None,
    "quad_interaction": True,
    "atom_edge_interaction": True,
    "edge_atom_interaction": True,
    "atom_interaction": True,
    "scale_basis": True,
}


In [4]:
GemNet = GemNetOC(**model_cfg)

# Count total number of parameters
trainable_params = sum(p.numel() for p in GemNet.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable_params}")

Trainable parameters: 37815808


## Testing GemNet-OC for energy/force regression

In [5]:
optimizer = optim.Adam(GemNet.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

# Training loop
def train():
    '''
    Defines the training procedure for the neural network
    '''
    for epoch in range(100): 
  
        # Create batch index (all atoms belong to structure 0)
        x1 = positions 
        batch = torch.zeros(len(x1), dtype=torch.long)

        # Build data objects
        data = Data(pos=x1, atomic_numbers=atomic_numbers, batch=batch)
        data.cell = torch.tensor(contcar.get_cell().array, dtype=torch.float32).unsqueeze(0) 
        data.pbc = torch.tensor([[True, True, True]])
        data.natoms = torch.tensor([len(positions)], dtype=torch.long)
        data.tags = torch.zeros(len(positions), dtype=torch.long)

        # Compute u_theta 
        output = GemNet(data)

        # Compute loss 
        force = torch.tensor([[0, 0, 1], [0, 0, -1]],  dtype=torch.float32)
        loss = loss_fn(output, force)

        print(f"Loss at epoch {epoch}: {loss.item()}")

        if loss.item() < 0.0001:
            break 

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


    return output

## Trainer

In [4]:
GemFlow = GemNetFlow(**model_cfg)

In [32]:
optimizer = optim.Adam(GemFlow.parameters(), 1e-3)
loss_fn = nn.MSELoss()

# Training loop
def train():
    '''
    Defines the training procedure for the neural network
    '''
    for epoch in range(100000): 
        
        x1 = positions 
        x0 = torch.randn_like(x1) 
        t = torch.rand(1)
    
        xt = (1 - t) * x0 + t * x1
        
        batch = torch.zeros(len(xt), dtype=torch.long)
        
        # Build data objects
        data = Data(pos=xt, atomic_numbers=atomic_numbers, batch=batch)
        data.cell = torch.tensor(contcar.get_cell().array, dtype=torch.float32).unsqueeze(0) 
        data.pbc = torch.tensor([[True, True, True]])
        data.natoms = torch.tensor([len(positions)], dtype=torch.long)
        data.t = t
        data.tags = torch.zeros(len(positions), dtype=torch.long)

        # Compute u_theta 
        u_theta = GemFlow(data)
    
        # Compute loss 
        loss = loss_fn(u_theta, x1 - x0)
        print(f"Loss at epoch {epoch}: {loss.item()}")

        if loss.item() < 0.01:
            break 

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

train()

Loss at epoch 0: 0.4234480857849121
Loss at epoch 1: 3835.405029296875
Loss at epoch 2: 404.5877685546875
Loss at epoch 3: 159.59239196777344
Loss at epoch 4: 895.8345336914062
Loss at epoch 5: 295.4034118652344
Loss at epoch 6: 1029.21435546875
Loss at epoch 7: 861.5261840820312
Loss at epoch 8: 125.82137298583984
Loss at epoch 9: 84.75593566894531
Loss at epoch 10: 554.4819946289062
Loss at epoch 11: 807.3316040039062
Loss at epoch 12: 477.702392578125
Loss at epoch 13: 42.16633987426758
Loss at epoch 14: 39.67424392700195
Loss at epoch 15: 136.39488220214844
Loss at epoch 16: 351.5264892578125
Loss at epoch 17: 289.28436279296875
Loss at epoch 18: 97.92791748046875
Loss at epoch 19: 12.94792652130127
Loss at epoch 20: 235.79266357421875
Loss at epoch 21: 13.131623268127441
Loss at epoch 22: 94.83700561523438
Loss at epoch 23: 15.168644905090332
Loss at epoch 24: 14.306962013244629
Loss at epoch 25: 20.128549575805664
Loss at epoch 26: 7.984925746917725
Loss at epoch 27: 30.737119674

KeyboardInterrupt: 

## Sampler

In [31]:
# Sampler function

def step(xt: Tensor, t_start: Tensor, t_end: Tensor) -> Tensor: 

    # Define data object 
    data = Data() 
    data.atomic_numbers = atomic_numbers
    data.batch = torch.zeros(len(x1), dtype=torch.long)
    data.cell = torch.tensor(contcar.get_cell().array, dtype=torch.float32).unsqueeze(0) 
    data.pbc = torch.tensor([[True, True, True]])
    data.natoms = torch.tensor([len(positions)], dtype=torch.long)
    data.tags = torch.zeros(len(positions), dtype=torch.long)


    #t_start = t_start.view(1, 1).expand(xt.shape[0], 1)

    midpoint_time = t_start + (t_end - t_start) / 2 # evalute the midpoint 

    data.t = t_start
    data.pos = xt
    intermediate_value = GemFlow(data) # compute the velocity at this point using the trained neural network
    updated_xt = xt + intermediate_value * (t_end - t_start) / 2 # move xt using computed velocity 

    data.t = midpoint_time
    data.pos = updated_xt
    midpoint_output = GemFlow(data) # compute vt again 
    
    return xt + (t_end - t_start) * midpoint_output # compute xt

GemFlow.eval()  # Disables dropout/batch norm and ensures no gradients
x1 = positions 
x0 = torch.randn_like(x1) 
x = step(xt=x0, t_start=torch.Tensor([0]), t_end=torch.Tensor([1]))

print(x0)
print(x)


tensor([[ 0.1967, -0.4065,  0.5062],
        [ 0.8594, -0.6319, -0.6370]])
tensor([[-0.0657,  0.0243, -0.3239],
        [ 1.1218, -1.0627,  0.1931]], grad_fn=<AddBackward0>)


In [28]:
# Sample from p1

gen_struc = Atoms("H2", positions=x.detach().numpy())

view(gen_struc)


<Popen: returncode: None args: ['/opt/anaconda3/envs/cfm/bin/python', '-m', ...>